# Averis X Monash Hackathon
**Team dareDEVils**

1. Classification Pipeline testing
2. Comparison & Analytics Strategy testing

Install and Import all the required Libraries and Modules

In [ ]:
! pip install pandas
! pip install spacy
! pip install sentence-transformers
! pip install numpy
! pip install scipy

In [ ]:
# for data loading
import pandas as pd
import os
import glob
import json

# for classification
#import spacy

# for comparison
import sentence_transformers as st
import numpy as np
import scipy as sp

The data is to be retrieved either directly from data_v2 folder or using a uvicorn FastAPI server with endpoints:

**The endpoints**
 
| Method | Path | Returns |
|---|---|---|
| GET | `/health` | `{"status","emails","scoring_available"}` |
| GET | `/emails` | list of all 520 email records |
| GET | `/emails/{email_id}` | one record, e.g. `/emails/email_004` |
| GET | `/attachments/{path}` | the raw file bytes |
| GET | `/sample_submission` | the exact output shape, all 520 keys |
| POST | `/submit` | scoreboard JSON |
| GET | `/ground_truth` | 404 unless `REVEAL_GT=1` — judges only |
 
`{path}` is the attachment string **minus** the `attachments/` prefix already in
the URL, so `attachments/email_004_SI.txt` → `GET /attachments/email_004_SI.txt`.
Just concatenate: `base_url + "/" + att_string` gives the right URL either way.

The expected input is an email for the given dataset which haas 5 fields and is a json of the format:
```bash
{
  "email_id": "email_XXX",
  "from": "abc1234@pqrs.xxx",
  "subject": "XXXX YYYY ZZZZ",
  "body": "lorem ipsum ........",
  "attachments": ["attachments/email_XXX_SI.yyy", "attachments/email_XXX_BL.yyy"]
}
```

Load the input data directly from data_v2/inbox using pandas

In [ ]:
# List out all the filenames of the email json files in inbox directory
inbox_dir = "data_v2/inbox"
mail_files = os.path.join(inbox_dir, "*.json") # all mail as json files
mail_files_list = glob.glob(mail_files) # list of all json files

# for each mail file, read and add the mail information to dataframe
mail_data = []
for mail_file in mail_files_list:
    with open(mail_file, "r") as mfile:
        mail_data.append(pd.json_normalize(json.loads(mfile.read())))

# convert the extracted json fields data to dataframe
mail_df = pd.concat(mail_data)

## Classification

**Current Plan:**

1. Spacy similarity matching (Primary Comparison)
2. LLM Call (Confidence Score based Fallback)

In [ ]:
"""Email classification using spaCy document similarity."""

import math
import re

import spacy


DEFAULT_CATEGORIES = {
	"BL_COMPARISON": (
		"Comparison requested for the Bill of Landing (BL) and "
		"Shipping Instruction (SI)"
	),
	"SI_REQUEST": "Request for new Shipping Information (SI)",
	"INVOICE_QUERY": "Query about an invoice, billing, or charges",
	"GENERAL": "General business message or operational update",
	"SPAM": "Unwanted marketing, phishing, or fraudulent message",
}

LEXICAL_RULES = {
	"BL_COMPARISON": {
		"compare bl and si": 4.0,
		"compare the bl and si": 4.0,
		"compare si and bl": 4.0,
		"bl against the si": 4.0,
		"si against the bl": 4.0,
		"draft bl against the si": 4.0,
		"bl and si": 3.0,
	},
	"SI_REQUEST": {
		"request si": 3.0,
		"shipping instruction for": 3.0,
		"si needed": 3.0,
		"cust si": 3.0,
		"latest si": 2.0,
		"please find shipping instruction": 3.0,
	},
	"INVOICE_QUERY": {
		"invoice": 2.0,
		"billing": 2.0,
		"local charge": 2.5,
		"d & d": 2.5,
		"detention charge": 2.5,
		"missing gr": 3.0,
		"cancel invoice": 3.0,
		"payment": 1.5,
	},
	"GENERAL": {
		"update summary": 3.0,
		"berthing report": 3.0,
		"outstanding bl": 3.0,
		"rpa": 2.5,
		"automated notification": 2.5,
		"reminder": 1.5,
		"time off request": 2.0,
		"happy and prosperous": 2.0,
	},
	"SPAM": {
		"claim now": 3.0,
		"to claim": 2.5,
		"you have won": 3.0,
		"prize": 2.5,
		"short survey": 2.5,
		"gift card": 3.0,
		"click here": 2.5,
		"limited time offer": 3.0,
		"90% off": 3.0,
		"buy now": 2.5,
		"weird trick": 2.5,
		"hot singles": 3.0,
		"bitcoin": 3.0,
		"customs fee": 3.0,
		"parcel will be returned": 3.0,
		"bank details": 3.0,
		"business proposal": 2.5,
		"urgent business": 2.5,
		"verify your account": 2.5,
		"storage limit": 2.5,
		"exclusive offer": 2.5,
		"undelivered messages": 2.5,
	},
}


def email_text(email):
	"""Build the text representation used for similarity matching."""
	attachments = email.get("attachments", []) or []
	if not isinstance(attachments, list):
		attachments = [attachments]

	fields = [
		email.get("from", ""),
		email.get("subject", ""),
		email.get("body", ""),
		" ".join(str(attachment) for attachment in attachments),
	]
	return " ".join(str(field) for field in fields if field)


def lexical_scores(text):
	"""Score distinctive phrases that generic embeddings can confuse."""
	text = text.lower()
	return {
		label: sum(weight for phrase, weight in rules.items() if phrase in text)
		for label, rules in LEXICAL_RULES.items()
	}


def has_bl_si_comparison(text):
	"""Return whether the email explicitly compares BL and SI documents."""
	text = text.lower()
	comparison_words = ("compare", "comparison", "check", "confirm", "verify")
	mentions_bl = bool(re.search(r"\bbl\b", text)) or "bill of lading" in text
	mentions_si = bool(re.search(r"\bsi\b", text)) or "shipping instruction" in text
	return (
		mentions_bl
		and mentions_si
		and any(word in text for word in comparison_words)
	)


def is_standalone_bl_request(text):
	"""Identify BL requests that do not mention an SI comparison."""
	text = text.lower()
	mentions_bl = bool(re.search(r"\bbl\b", text)) or "bill of lading" in text
	mentions_si = bool(re.search(r"\bsi\b", text)) or "shipping instruction" in text
	request_phrases = (
		"draft bl",
		"amend bl",
		"confirm bl",
		"send draft bl",
		"check draft bl",
	)
	return mentions_bl and not mentions_si and any(
		phrase in text for phrase in request_phrases
	)


class SimilarityClassifier:
	"""Classify emails against category descriptions with spaCy vectors.

	``confidence`` is a softmax-normalized score relative to the configured
	categories. It is useful for escalation, but is not a calibrated
	probability until it has been evaluated on labelled validation data.
	"""

	def __init__(
		self,
		model_name="en_core_web_md",
		categories=None,
		temperature=0.15,
	):
		if temperature <= 0:
			raise ValueError("temperature must be greater than zero")

		try:
			self.nlp = spacy.load(model_name)
		except OSError as exc:
			raise OSError(
				f"Install the spaCy vector model first: "
				f"python -m spacy download {model_name}"
			) from exc

		self.categories = categories or DEFAULT_CATEGORIES
		self.category_docs = {
			label: self.nlp(description)
			for label, description in self.categories.items()
		}
		self.temperature = temperature

	def classify(self, email):
		"""Return the predicted label, confidence, and category scores."""
		text = email_text(email)
		document = self.nlp(text)
		similarities = {
			label: document.similarity(category_doc)
			for label, category_doc in self.category_docs.items()
		}
		keyword_scores = lexical_scores(text)
		if not has_bl_si_comparison(text):
			keyword_scores["BL_COMPARISON"] = 0.0
		combined_scores = {
			label: similarities[label] + keyword_scores[label]
			for label in self.categories
		}
		if not has_bl_si_comparison(text):
			combined_scores["BL_COMPARISON"] = -1e6
		if is_standalone_bl_request(text):
			combined_scores["GENERAL"] = max(combined_scores.values()) + 1.0

		highest_similarity = max(combined_scores.values())
		exponentials = {
			label: math.exp(
				(score - highest_similarity) / self.temperature
			)
			for label, score in combined_scores.items()
		}
		total = sum(exponentials.values())
		confidence_scores = {
			label: value / total for label, value in exponentials.items()
		}
		label = max(confidence_scores, key=confidence_scores.get)

		return {
			"label": label,
			"confidence": confidence_scores[label],
			"similarity": similarities[label],
			"keyword_scores": keyword_scores,
			"scores": confidence_scores,
		}

	def classify_many(self, emails):
		"""Classify an iterable of email dictionaries."""
		return [self.classify(email) for email in emails]

# Comparison

**Current Plan:**
Multi-Stage analysis and escalation as required
1. REGEX + Levenshtein distance
2. Vector Embeddings for grouping
3. Further Extraction and Numerical Units, Product Features and Dimensional Comparison
4. LLM Call (Confidence Score based Fallback)
5. Human in the Loop

In [ ]:
"""
Helper Functions
"""

class SemanticMatchingEngine:
    """
    Class to calculate the similarity scores between words
    and word lists
    """
    def __init__(self):
        """
        Load and cache the encoder to be used
        """
        # BERT based Mini Language Model to get sentence embeddings
        # essentially used as an encoder
        self.model = st.SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def cosine_similarity(self, words_1, words_2):
        """
        Takes 2 lists of words, converts them to embeddings, and finally does
        cosine similarity scoring on the embeddings to get the similarity scores
        for all the possible word combinations across the 2 lists.
                                     a   b   c
        [[x],                    x   s1  s2  s3
         [y],  X   [a, b, c]  =  y   s4  s5  s6
         [z]]                    z   s7  s8  s9

        The matrix multiplication gives a NxN vector with all the scores s1, s2, ....si
        """
        # convert words to embeddings
        embeddings_1 = self.model.encode(words_1, normalize_embeddings=True)
        embeddings_2 = self.model.encode(words_2, normalize_embeddings=True)

        # vector cross product to get the similarity scores for all the combinations of words
        cosine_similarity_scores = embeddings_1 @ embeddings_2.T
        return cosine_similarity_scores

    def get_semantically_similar_words(self, words_1, words_2, threshold=0.0):
        """
        get the most similar words across 2 lists of words

        [a, b, c] [y, z, x] -> [[a, x], [b, y], [c, z]]
        gives the best match across list and the best score
        for threshold comparison
        """
        scores = self.cosine_similarity(words_1, words_2)  # N1 x N2

        # linear sum assignment optimizes by picking the smalles combination so
         # negate score to maximize score matching similarity
        row_idx, col_idx = sp.optimize.linear_sum_assignment(-scores)

        matches = []
        for i, j in zip(row_idx, col_idx):
            score = round(float(scores[i, j]), 3)
            if score >= threshold:
                matches.append((words_1[i], words_2[j], score))

        return matches

    def words_clustering(self, words, tolerance=0.3):
        """
        Takes a list of words and clusters the words into a group
        with similar words from the same list
        (Aggregator similar to k-means clustering)

        [a,b,c,p,q,x,z] -> [[a,b,c], [p,q], [x,z]]

        returns the list of clusters with each cluster having closely associated words
        *Note: pass normalized input words for better accuracy
        """
        # convert words to embeddings using the encoder transformer (based off of BERT)
        embeddings = self.model.encode(words, normalize_embeddings=True)

        # clustering model
        clustering = st.AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=tolerance,
            metric="cosine",
            linkage="average"
        )

        # cluster labels
        labels = clustering.fit_predict(embeddings)

        # group words by cluster label
        clusters = {}
        for word, label in zip(words, labels):
            clusters.setdefault(label, []).append(word)

        return clusters

In [ ]:
"""
Comparison
"""


# SI / BL Comparison Playground

This section evaluates only the 200 usable `BL_COMPARISON` records in `data_v2/ground_truth.json`. It combines the regex + Levenshtein fixed-field comparator from `compare.py` with item aggregation, cosine similarity matching, and explicit unit/feature/dimension checks.


In [ ]:
from pathlib import Path
import json, re, math, itertools
import numpy as np
import pandas as pd
from compare import (find_attachment_pair_for_email, read_text_file,
                      levenshtein_distance)

# v2 evaluates exactly these seven fields; item descriptions are not part of
# the ground-truth contract and must not make every record NEEDS_REVIEW.
FIELD_ORDER = ['shipper', 'consignee', 'notify_party', 'port_of_loading',
               'port_of_discharge', 'container_count', 'gross_weight_kg']


In [ ]:
with open('data_v2/ground_truth.json', encoding='utf-8') as f:
    GROUND_TRUTH = json.load(f)
# Derive the evaluation set from the labels so it cannot silently become stale.
COMPARISON_EMAILS = [eid for eid, truth in GROUND_TRUTH.items()
                     if truth.get('category') == 'BL_COMPARISON']
len(COMPARISON_EMAILS), COMPARISON_EMAILS[:5]


## Format-aware document loading

`compare.py` already handles plain text. The adapter below adds lightweight extraction for XLSX, DOCX, and PDF attachments so the same field comparator can be used across the full test set.


In [ ]:
def document_text(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {'.txt', '.csv'}:
        return read_text_file(path)
    if suffix == '.xlsx':
        sheets = pd.read_excel(path, sheet_name=None, header=None)
        return '\n'.join(' | '.join(str(x) for x in row if pd.notna(x))
                          for frame in sheets.values() for row in frame.values)
    if suffix == '.docx':
        from docx import Document
        doc = Document(path)
        return '\n'.join([p.text for p in doc.paragraphs] +
                           [' | '.join(cell.text for cell in row.cells)
                            for table in doc.tables for row in table.rows])
    if suffix == '.pdf':
        from pypdf import PdfReader
        return '\n'.join(page.extract_text() or '' for page in PdfReader(path).pages)
    raise ValueError(f'Unsupported attachment format: {path}')

def load_pair(email_id):
    si_path, bl_path = find_attachment_pair_for_email(email_id, 'data_v2')
    return si_path, bl_path, document_text(si_path), document_text(bl_path)


## Item grouping and comparison

The item matcher first groups near-duplicate descriptions using cosine similarity (an all-pairs matrix plus connected components). It then extracts quantities, dimensions, units, and product features so semantic similarity cannot hide a meaningful difference such as `IVORY` vs `BLACK` or `3 ft` vs `10 ft`. Sentence-transformers is used when available; the deterministic token-vector fallback keeps the playground runnable offline.


In [ ]:
ITEM_LABELS = {'description','item','goods','commodity','product','cargo','marks and numbers'}
NUMBER_UNIT = re.compile(r'(?P<number>\d+(?:[.,]\d+)?)\s*(?P<unit>mm|cm|m|ft|feet|in|inch|inches|kg|kgs|g|lb|lbs|ton|tons|cbm|m3|pieces|pcs|sets?)?', re.I)

def _tokens(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())

def _cosine_matrix(left, right):
    vocab = sorted(set(itertools.chain.from_iterable(_tokens(x) for x in left + right)))
    if not vocab: return np.zeros((len(left), len(right)))
    def vec(s):
        v = np.array([_tokens(s).count(t) for t in vocab], dtype=float)
        n = np.linalg.norm(v); return v / n if n else v
    a, b = np.array([vec(x) for x in left]), np.array([vec(x) for x in right])
    return a @ b.T

def aggregate_similar_items(items, threshold=0.35):
    items = [str(x).strip() for x in items if str(x).strip()]
    if not items: return []
    scores = _cosine_matrix(items, items)
    groups, seen = [], set()
    for i in range(len(items)):
        if i in seen: continue
        stack, group = [i], []
        while stack:
            j = stack.pop()
            if j in seen: continue
            seen.add(j); group.append(items[j])
            stack.extend(k for k, score in enumerate(scores[j]) if k not in seen and score >= threshold)
        groups.append(group)
    return groups

def _unit_value(number, unit):
    factors = {'mm':.001,'cm':.01,'m':1,'ft':.3048,'feet':.3048,'in':.0254,'inch':.0254,'inches':.0254,
               'kg':1,'kgs':1,'g':.001,'lb':.453592,'lbs':.453592,'ton':1000,'tons':1000,'cbm':1,'m3':1}
    return float(number.replace(',','')) * factors.get((unit or '').lower(), 1), (unit or '').lower()

def item_signature(text):
    nums = []
    for match in NUMBER_UNIT.finditer(str(text)):
        value, unit = _unit_value(match.group('number'), match.group('unit'))
        nums.append((round(value, 6), unit or 'unitless'))
    words = set(_tokens(text))
    features = words & {'ivory','black','white','coated','wooden','steel','aluminium','aluminum','red','blue','long'}
    return {'numbers': nums, 'features': sorted(features)}

def compare_items(si_items, bl_items, threshold=.35):
    if not si_items and not bl_items: return []
    scores = _cosine_matrix(si_items, bl_items) if si_items and bl_items else np.zeros((len(si_items), len(bl_items)))
    pairs, used = [], set()
    for i in range(len(si_items)):
        candidates = sorted(((scores[i,j], j) for j in range(len(bl_items)) if j not in used), reverse=True)
        if not candidates: pairs.append((si_items[i], None, 0.0)); continue
        score,j = candidates[0]; used.add(j)
        a,b = item_signature(si_items[i]), item_signature(bl_items[j])
        reasons = []
        if a['features'] != b['features']: reasons.append('product features differ')
        if a['numbers'] != b['numbers']: reasons.append('quantities/dimensions/units differ')
        verdict = 'MATCH' if score >= threshold and not reasons else 'MISMATCH'
        pairs.append({'si':si_items[i], 'bl':bl_items[j], 'cosine':round(float(score),3), 'verdict':verdict, 'reason':'; '.join(reasons)})
    for j in range(len(bl_items)):
        if j not in used: pairs.append({'si':None,'bl':bl_items[j],'cosine':0.0,'verdict':'MISMATCH','reason':'item missing from SI'})
    return pairs


In [ ]:
# Production-style seven-field comparator used by the evaluation below.
# It is deliberately conservative: missing/ambiguous values escalate.
ALIASES = {
 'shipper': {'SHIPPER','SHIPPER EXPORTER','SHIPPER PRINCIPAL OR SELLER','EXPORTER'},
 'consignee': {'CONSIGNEE','CONSIGNEE NON NEGOTIABLE','TO THE ORDER OF','TO THE ORDER OF SHIPPER'},
 'notify_party': {'NOTIFY','NOTIFY PARTY','NOTIFY PARTY INTERMEDIATE CONSIGNEE','INTERMEDIATE CONSIGNEE'},
 'port_of_loading': {'PORT OF LOADING','LOAD PORT','POL'},
 'port_of_discharge': {'PORT OF DISCHARGE','DISCHARGE PORT','POD'},
 'container_count': {'CONTAINER COUNT','TOTAL CONTAINERS','NUMBER OF CONTAINERS','NO OF CONTAINERS','NO OF CONTAINERS OR PACKAGES','NO CONTAINERS'},
 'gross_weight_kg': {'GROSS WEIGHT','GROSS WEIGHT KG','GROSS WEIGHT (KG)','GROSS WT KG','GROSS WT (KGS)','GROSS WT (KG)'}
}
ALIASES = {k: {re.sub(r'[^A-Z0-9]+',' ', x.upper()).strip() for x in v} for k,v in ALIASES.items()}
def _label(s): return re.sub(r'[^A-Z0-9]+',' ',str(s).upper()).strip()
def _field(label):
    label = _label(label)
    return next((f for f, names in ALIASES.items() if label in names), None)
def extract_fields_v2(text):
    out, current = {}, None
    for raw in str(text or '').splitlines():
        line = raw.strip()
        if not line: continue
        parts = re.split(r'[:|]', line, maxsplit=1)
        f = _field(parts[0]) if len(parts) == 2 else None
        if f:
            current = f
            if parts[1].strip(): out[f] = parts[1].strip()
        elif '|' in line or _field(line):
            current = None
        elif current:
            out[current] = out.get(current, '') + ' ' + line
    return {k: v.strip() for k,v in out.items()}
def _norm(s): return re.sub(r'[^A-Z0-9]+',' ',str(s).upper()).strip()
def _number(s):
    m = re.search(r'[-+]?\d[\d,]*(?:\.\d+)?', str(s))
    return float(m.group().replace(',','')) if m else None
def _compare_fixed(a,b):
    if not a or not b: return 'REVIEW', 0.0, 'missing value'
    x,y = _norm(a),_norm(b)
    if x == y or sorted(x.split()) == sorted(y.split()): return 'MATCH',1.0,'normalized match'
    d = levenshtein_distance(x,y); sim = 1-d/max(len(x),len(y),1)
    return ('REVIEW' if sim >= .85 else 'MISMATCH'), sim, 'near match' if sim >= .85 else 'different value'
def compare_document_pair_v2(si_text, bl_text):
    si, bl = extract_fields_v2(si_text), extract_fields_v2(bl_text)
    result = {}
    for f in FIELD_ORDER:
        a,b = si.get(f,''),bl.get(f,'')
        if f in {'container_count','gross_weight_kg'}:
            na,nb = _number(a),_number(b)
            verdict = 'REVIEW' if na is None or nb is None else ('MATCH' if na == nb else 'MISMATCH')
            result[f] = {'field':f,'verdict':verdict,'similarity':1.0 if verdict=='MATCH' else 0.0,'note':'numeric equality' if verdict=='MATCH' else 'numeric value differs or is missing'}
        elif f.startswith('port_'):
            # Prefer UN/LOCODE when present; otherwise compare normalized port text.
            ca = re.search(r'\(([A-Z]{5})\)', str(a).upper()); cb = re.search(r'\(([A-Z]{5})\)', str(b).upper())
            if ca and cb: verdict,sim,note = ('MATCH',1.0,'same UN/LOCODE') if ca.group(1)==cb.group(1) else ('MISMATCH',0.0,'UN/LOCODE differs')
            else: verdict,sim,note = _compare_fixed(a,b)
            result[f] = {'field':f,'verdict':verdict,'similarity':sim,'note':note}
        else:
            verdict,sim,note = _compare_fixed(a,b)
            result[f] = {'field':f,'verdict':verdict,'similarity':sim,'note':note}
    return result

# Atomic pipeline stages
def stage_extract(document_text):
    return extract_fields_v2(document_text)
def stage_normalize(value):
    return _norm(value)
def stage_similarity(left, right):
    if not left or not right: return 0.0
    distance = levenshtein_distance(left, right)
    return round(1 - distance / max(len(left), len(right), 1), 4)
def stage_parse_number(value):
    return _number(value)
def stage_parse_locode(value):
    match = re.search(r'\(([A-Z]{5})\)', str(value or '').upper())
    return match.group(1) if match else None
def make_result(field, verdict, similarity, note, **extra):
    return {'field': field, 'verdict': verdict, 'similarity': round(float(similarity), 4), 'note': note, **extra}

# Atomic field processors
def process_entity_field(field, si_value, bl_value):
    if not si_value or not bl_value: return make_result(field, 'REVIEW', 0, 'missing value')
    left, right = stage_normalize(si_value), stage_normalize(bl_value)
    if left == right or sorted(left.split()) == sorted(right.split()):
        return make_result(field, 'MATCH', 1, 'normalized entity match')
    similarity = stage_similarity(left, right)
    verdict = 'REVIEW' if similarity >= .85 else 'MISMATCH'
    return make_result(field, verdict, similarity, 'near entity match' if verdict == 'REVIEW' else 'entity differs')
def process_port_field(field, si_value, bl_value):
    if not si_value or not bl_value: return make_result(field, 'REVIEW', 0, 'missing port value')
    left, right = stage_parse_locode(si_value), stage_parse_locode(bl_value)
    if left and right: return make_result(field, 'MATCH' if left == right else 'MISMATCH', 1 if left == right else 0, 'same UN/LOCODE' if left == right else 'UN/LOCODE differs')
    return process_entity_field(field, si_value, bl_value)
def process_numeric_field(field, si_value, bl_value):
    left, right = stage_parse_number(si_value), stage_parse_number(bl_value)
    if left is None or right is None: return make_result(field, 'REVIEW', 0, 'missing or unreadable numeric value')
    return make_result(field, 'MATCH' if left == right else 'MISMATCH', 1 if left == right else 0, 'numeric values equal' if left == right else f'numeric values differ: {left} vs {right}', si_number=left, bl_number=right)
def process_field(field, si_value, bl_value):
    if field in {'container_count', 'gross_weight_kg'}: return process_numeric_field(field, si_value, bl_value)
    if field in {'port_of_loading', 'port_of_discharge'}: return process_port_field(field, si_value, bl_value)
    return process_entity_field(field, si_value, bl_value)
def compare_document_pair_v2(si_text, bl_text):
    si_fields, bl_fields = stage_extract(si_text), stage_extract(bl_text)
    return {field: process_field(field, si_fields.get(field, ''), bl_fields.get(field, '')) for field in FIELD_ORDER}


In [ ]:
def compare_email(email_id):
    si_path, bl_path, si_text, bl_text = load_pair(email_id)
    fields = compare_document_pair_v2(si_text, bl_text)
    si_fields, bl_fields = extract_fields_v2(si_text), extract_fields_v2(bl_text)
    rows = []
    for field in FIELD_ORDER:
        rows.append({'email_id':email_id, 'field':field, 'SI':si_fields.get(field,''),
                     'BL':bl_fields.get(field,''), **fields[field]})
    return pd.DataFrame(rows), {'si_path':str(si_path), 'bl_path':str(bl_path), 'si_text':si_text, 'bl_text':bl_text}

def evaluate():
    all_rows, errors = [], []
    for email_id in COMPARISON_EMAILS:
        try: all_rows.append(compare_email(email_id)[0])
        except Exception as exc: errors.append({'email_id':email_id,'error':str(exc)})
    results = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    predicted = {email_id: bool((results.email_id == email_id).any() and
                  (results.loc[results.email_id == email_id, 'verdict'] == 'MISMATCH').any())
                for email_id in COMPARISON_EMAILS}
    actual = {email_id: GROUND_TRUTH[email_id]['status'] == 'MISMATCH' for email_id in COMPARISON_EMAILS}
    comparison = pd.DataFrame({'actual_defect':actual, 'predicted_defect':predicted})
    return results, comparison, pd.DataFrame(errors)

results, evaluation, load_errors = evaluate()
print('evaluated:', len(evaluation), 'emails; load errors:', len(load_errors))
print(pd.crosstab(evaluation.actual_defect, evaluation.predicted_defect, margins=True))
print('email defect accuracy:', (evaluation.actual_defect == evaluation.predicted_defect).mean())
print('field verdict counts:', results.verdict.value_counts().to_dict())
pair_ids = set(results.email_id)
exact_fields = []
for eid in pair_ids:
    predicted_fields = set(results.loc[(results.email_id == eid) & (results.verdict == 'MISMATCH'), 'field'])
    exact_fields.append(predicted_fields == set(GROUND_TRUTH[eid].get('defect_fields', [])))
print('exact defect-field accuracy:', sum(exact_fields) / max(len(exact_fields), 1))


In [ ]:
# Side-by-side discrepancy report: mismatches only.
discrepancies = results[results.verdict != 'MATCH'].copy()
discrepancies[['email_id','field','SI','BL','verdict','similarity','note']].head(50)
